# 第 5 讲：GPU 基础

这是 CS336 Lecture 05 的可执行中文学习版。

- 官方来源：`external/cs336-lectures/lecture_05.pdf`
- 官方形式：PDF lecture
- 英文标题：GPUs

这份 notebook 是学习版，不是逐字翻译。它按官方讲义的结构和节奏整理主线，并在适合的位置加入小实验。


## 0. 本讲你要抓住什么

这一讲把 GPU 从黑盒拆开：SM、warp、register、shared memory、HBM、roofline，以及为什么 FlashAttention 能快。

学习方式：

1. 先读每节的中文解释。
2. 运行对应代码 cell。
3. 改一个参数，观察输出如何变化。
4. 最后用检查点问题自测。


## 1. 本讲路线图

先理解硬件结构，再理解性能瓶颈，最后用 FlashAttention 串起来。


In [1]:
lecture = 5
title = 'GPUs'
source = 'external/cs336-lectures/lecture_05.pdf'
print(f"Lecture {lecture}: {title}")
print("source:", source)


Lecture 5: GPUs
source: external/cs336-lectures/lecture_05.pdf


## 2. GPU 为什么适合深度学习

GPU 用大量并行线程和高带宽内存服务矩阵乘法、attention、elementwise kernel。


## 3. Memory hierarchy

register 最快最小，shared memory 局部共享，HBM 大但慢；优化常常是减少 HBM 往返。


## 4. Warp 与 occupancy

warp 是 32 个线程的执行单位；occupancy 影响隐藏内存延迟的能力。


## 5. Roofline

性能上限取决于 compute 峰值和 memory bandwidth，两者由 arithmetic intensity 连接。


## 6. FlashAttention 直觉

不显式 materialize 巨大的 attention matrix，而是分块计算，减少 HBM 读写。


## 动手实验 1：Roofline 玩具模型

先直接运行，再改一个数字或字符串。


In [2]:
peak_flops = 1_000
bandwidth = 100
for intensity in [0.1, 1, 5, 20]:
    attainable = min(peak_flops, bandwidth * intensity)
    bound = "memory" if bandwidth * intensity < peak_flops else "compute"
    print("intensity", intensity, "attainable", attainable, "bound", bound)


intensity 0.1 attainable 10.0 bound memory
intensity 1 attainable 100 bound memory
intensity 5 attainable 500 bound memory
intensity 20 attainable 1000 bound compute


## 动手实验 2：Occupancy 粗算

先直接运行，再改一个数字或字符串。


In [3]:
max_registers_per_sm = 65536
max_warps_per_sm = 64
threads_per_block = 128
registers_per_thread = 160
registers_per_block = threads_per_block * registers_per_thread
blocks = max_registers_per_sm // registers_per_block
warps = blocks * threads_per_block / 32
print("blocks per SM:", blocks)
print("warps per SM:", warps)
print("occupancy:", warps / max_warps_per_sm)


blocks per SM: 3
warps per SM: 12.0
occupancy: 0.1875


## 动手实验 3：Attention matrix 的内存压力

先直接运行，再改一个数字或字符串。


In [4]:
for seq in [2048, 8192, 32768]:
    bytes_fp16 = seq * seq * 2
    print(seq, "attention matrix GB:", bytes_fp16 / 1e9)


2048 attention matrix GB: 0.008388608
8192 attention matrix GB: 0.134217728
32768 attention matrix GB: 2.147483648


## 今日检查点

1. 能解释 HBM、shared memory、register 的区别。
2. 能用 roofline 判断一个操作更像 memory-bound 还是 compute-bound。
3. 能说出 FlashAttention 为什么不是改数学结果，而是改数据移动方式。

如果这些能讲清楚，这一讲的第一轮学习就完成了。
